### Imports

In [9]:
import numpy as np
import math
from scipy.stats import norm, t
from scipy import stats

### Part 1

In [10]:
n = 100

U = np.random.uniform(size=n)
X = np.exp(U)

mean = X.mean()
s = X.std(ddof=1)
se = s / np.sqrt(n)

z = 1.96
ci_low  = mean - z * se
ci_high = mean + z * se


print(mean)
print(s)
print(se)
print(ci_low, ci_high)

1.706868968017574
0.4593314335130747
0.04593314335130747
1.6168400070490112 1.7968979289861366


### Part 2

In [11]:
n_total_evaluations = 100
n_pairs = n_total_evaluations // 2 

u = np.random.rand(n_pairs)

y1 = np.exp(u)
y2 = np.exp(1 - u)

y = (y1 + y2) / 2

point_estimate = np.mean(y)

sem = stats.sem(y)
ci = stats.t.interval(0.95, df=n_pairs-1, loc=point_estimate, scale=sem)

print(f"Point Estimate: {point_estimate:.6f}")
print(f"95% Confidence Interval: ({ci[0]:.6f}, {ci[1]:.6f})")

Point Estimate: 1.725473
95% Confidence Interval: (1.705429, 1.745516)


### part 3

In [12]:
n = 100
U = np.random.uniform(size=n)
X = np.exp(U)
Z = U
c = -np.cov(X, Z)[0, 1] / np.var(Z, ddof=1)
Y = X + c * (Z - 0.5)
mean = Y.mean()
s = Y.std(ddof=1)
se = s / np.sqrt(n)
ci_low  = mean - z * se
ci_high = mean + z * se

print(mean)
print(s)
print(se)
print(ci_low, ci_high)

1.7225333580532969
0.06224221569067368
0.006224221569067368
1.710333883777925 1.7347328323286688


### part 4

In [13]:
k = 20
m = 5
n = k * m
strata = np.arange(k).reshape(-1, 1)
U = (strata + np.random.uniform(size=(k, m))) / k
X = np.exp(U)
mean = X.mean()
se = np.sqrt(X.var(axis=1, ddof=1).sum() / (k**2 * m))
ci_low  = mean - z * se
ci_high = mean + z * se

print(mean)
print(se)
print(ci_low, ci_high)

1.7163671822146256
0.002886285743112712
1.7107100621581246 1.7220243022711266


### part 5

In [14]:
class BlockingSystem():
    def __init__(self, arrival_sampler, service_mean=8, num_servers=10, num_customers=10*10000):
        self.arrival_sampler = arrival_sampler
        self.service_mean = service_mean
        self.num_servers = num_servers
        self.num_customers = num_customers
        self.time = 0
        self.servers = [(0, 0)] * num_servers # (start time, service time)
        self.blocked = 0
        self.sum_service_times = 0

    def sample_customer(self):
        arrival_delta = self.arrival_sampler()
        service_time = np.random.exponential(self.service_mean)
        
        self.sum_service_times += service_time
        self.time += arrival_delta

        return arrival_delta, service_time, self.time 

    def start_processing(self, customer, server):
        arrival_delta, service_time, arrived_timestamp = customer
        start_time = arrived_timestamp
        
        self.servers[server] = [start_time, service_time]

    def get_first_available_server(self):
        for i in range(len(self.servers)):
            if (self.servers[i][0] + self.servers[i][1] <= self.time):
                return i
        
        return -1

    
    def simulate(self):
        while self.num_customers > 0:
            customer = self.sample_customer()
            server = self.get_first_available_server()

            if (server == -1):
                self.blocked += 1
            else:
                self.start_processing(customer, server)

            self.num_customers -= 1

        return self.blocked, self.sum_service_times

def erlang_b_formula(arrival_mean, service_mean, m):
    A = arrival_mean * service_mean
    den = 0
    for i in range(m+1):
        den += (A**i / math.factorial(i))

    return ((A**m)/math.factorial(m))/den
        
# We conduct 100 independent blocking system experiments. 
# For each experiment, X[i] represents our estimate of the blocking rate, 
# and Z[i] represents the total service time (control variable). 
# E(Z) = mu_Z = num_customers*service_mean.

# Service times that are too long keep the servers busy for a longer period, 
# increasing the number of blockages so there is a positive correlation between X and Z. 
# This property allows us to decrease the variance through corrected observations,
# Y[i] = X[i] + c*(Z[i]-mu_Z), where the coefficient c is selected to minimize the variance Var(Y).

n_runs        = 100
num_customers = 10000
arrival_mean  = 1
service_mean  = 8
mu_Z          = num_customers * service_mean  # media nota di Z

X = np.zeros(n_runs)   # blocking rate per run
Z = np.zeros(n_runs)   # sum service times per run

for i in range(n_runs):
    bs = BlockingSystem(
        lambda: np.random.exponential(arrival_mean),
        service_mean, 10, num_customers
    )
    blocked, sum_st = bs.simulate()
    X[i] = blocked / num_customers
    Z[i] = sum_st

# optimal coeff 
c = -np.cov(X, Z)[0, 1] / np.var(Z, ddof=1)

Y = X + c * (Z - mu_Z)
z_val = 1.96

# crude MC
mean_mc = X.mean()
se_mc   = X.std(ddof=1) / np.sqrt(n_runs)

# using CV
mean_cv = Y.mean()
se_cv   = Y.std(ddof=1) / np.sqrt(n_runs)

print(f"Crude MC: {mean_mc:.5f}  CI: [{mean_mc - z_val*se_mc:.5f}, {mean_mc + z_val*se_mc:.5f}]")
print(f"CV:       {mean_cv:.5f}  CI: [{mean_cv - z_val*se_cv:.5f}, {mean_cv + z_val*se_cv:.5f}]")
print(f"Var MC:   {X.var(ddof=1):.6f}")
print(f"Var CV:   {Y.var(ddof=1):.6f}")
print(f"c: {c}")

Crude MC: 0.12147  CI: [0.12033, 0.12261]
CV:       0.12159  CI: [0.12063, 0.12256]
Var MC:   0.000034
Var CV:   0.000024
c: -4.055120252026665e-06


### Part 7

In [15]:
def compare_event_estimators():
    a_values = [2.0, 4.0]
    n_samples_list = [10000, 100000]
    sigma2_values = [0.5, 1.0]
    
    print(f"{'a':<5} | {'N':<8} | {'sigma2':<6} | {'Crude MC':<10} | {'Imp. Samp':<10} | {'Efficiency (Var)'}")
    print("-" * 75)

    for a in a_values:
        for n in n_samples_list:
            for s2 in sigma2_values:
                # Crude Monte Carlo (CMC)
                z = np.random.normal(0, 1, n)
                crude_indicator = (z > a)
                p_crude = np.mean(crude_indicator)
                var_crude = np.var(crude_indicator) / n
                
                # Importance Sampling (IS)
                # Sampling from a shifted proposal distribution g(x) ~ N(a, sigma^2)
                x = np.random.normal(a, np.sqrt(s2), n)
                
                # Calculate weights: likelihood ratio f(x)/g(x)
                f_x = norm.pdf(x, 0, 1)
                g_x = norm.pdf(x, a, np.sqrt(s2))
                weights = f_x / g_x
                
                # Weighted estimator
                is_values = (x > a) * weights
                p_is = np.mean(is_values)
                var_is = np.var(is_values) / n
                
                # Efficiency: Ratio of variances (Var_CMC / Var_IS)
                efficiency = var_crude / var_is
                
                print(f"{a:<5} | {n:<8} | {s2:<6} | {p_crude:<10.5f} | {p_is:<10.5f} | {efficiency:.2f}x")


compare_event_estimators()

a     | N        | sigma2 | Crude MC   | Imp. Samp  | Efficiency (Var)
---------------------------------------------------------------------------
2.0   | 10000    | 0.5    | 0.02320    | 0.02294    | 29.18x
2.0   | 10000    | 1.0    | 0.02170    | 0.02286    | 17.66x
2.0   | 100000   | 0.5    | 0.02218    | 0.02280    | 27.90x
2.0   | 100000   | 1.0    | 0.02320    | 0.02291    | 18.50x
4.0   | 10000    | 0.5    | 0.00000    | 0.00003    | 0.00x
4.0   | 10000    | 1.0    | 0.00000    | 0.00003    | 0.00x
4.0   | 100000   | 0.5    | 0.00005    | 0.00003    | 16845.05x
4.0   | 100000   | 1.0    | 0.00002    | 0.00003    | 4447.97x


### part 8

In [16]:
def sample_exp(lam, n):
    U = np.random.uniform(size=n)
    Y = -np.log(1 - U * (1 - np.exp(-lam))) / lam
    
    g = lam * np.exp(-lam * Y) / (1 - np.exp(-lam))
    weights = np.exp(Y) / g
    
    return weights.mean(), weights.var(ddof=1)

n = 10000

for lam in [0.5, 1.0, 2.0, 5.0]:
    est, var = sample_exp(lam, n)
    print(f"lambda={lam:.1f}  estimate={est:.6f}  Var={var:.6f}")

print()
print(np.e - 1)

lambda=0.5  estimate=1.720593  Var=0.575626
lambda=1.0  estimate=1.710452  Var=1.079181
lambda=2.0  estimate=1.685629  Var=2.741542
lambda=5.0  estimate=1.702790  Var=27.738093

1.718281828459045


In [17]:
# part 9

# Pareto has density f(x) = k/beta * (x/beta)^(-k-1) and the first moment distribution has density g(x) = (k-1)/β * (x/β)^(-k)

# The IS weight is f(x) / g(x) = k/(k-1) * beta/x

# So the IS estimate of the mean is:
# E_g[ x * f(x)/g(x) ] = E_g[ x * k/(k-1) * beta/x ]
#                         = E_g[ k*beta/(k-1) ]
#                        = k*beta/(k-1)

#  Which is formula for the Pareto mean => the variance is zero